In [ ]:
import pickle
import numpy as np

from src.linear_correlations import CorrelationGenerator
from src.resnet.model_with_embedding import FiniteResNetWithEmbedding
from src.resnet.activations import linear, linear_der

MAIN_SEED = 42
np.random.seed(MAIN_SEED)

In [ ]:
d_in = 3
d_out = 3

N = 1

x_scale = 0.8
x_true = np.random.randn(d_in)
x_true = x_scale*(x_true/np.linalg.norm(x_true))

A_true = np.random.randn(d_out, d_in)
noise_level = 0.05
y_true = A_true @ x_true + noise_level * np.random.randn(d_out)

In [ ]:
x_true, y_true, A_true

Training Parameters

In [ ]:
K = 15
eta_0 = 0.1

Infinite Model

In [ ]:
infinite_model = CorrelationGenerator(
    x=x_true,
    y=y_true,
    sigma_u=1, 
    sigma_v=1, 
    a=1, 
    eta_u=eta_0, 
    eta_v=eta_0, 
    sigma_we=1, 
    sigma_wu=1,
    K=K,
    num_s=100
    )

In [ ]:
result = infinite_model.run()

Finite Models (single large training)

In [ ]:
def train_track(model, X, Y, K, eta, eta_u, eta_v):
    losses = []
    H_track = []
    B_track = []
    out_track = []
    
    for k in range(K):
        loss, H, B, out = model.step(X, Y, eta, eta_u, eta_v, track=True)
        losses.append(loss)
        
        H_track.append(H.copy())
        B_track.append(B.copy())
        out_track.append(out.copy())
            
        print(f"[{k+1}/{K}] loss = {loss:.5f}")
        
    return np.array(losses), np.array(H_track), np.array(B_track), np.array(out_track)

In [ ]:
max_D = 1000
rootW_e = np.random.randn(max_D, d_in)
rootW_u = np.random.randn(max_D, d_out)

In [ ]:
with open(f"data/exp_track_single_trajectory/W_e_maxD{max_D}_din{d_in}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:   # note "wb" = write binary
    pickle.dump(rootW_e, f)
with open(f"data/exp_track_single_trajectory/W_u_maxD{max_D}_dout{d_out}_MAIN_SEED{MAIN_SEED}.pkl", "wb") as f:   # note "wb" = write binary
    pickle.dump(rootW_u, f)

In [ ]:
D, M, L = 1000,1000, 200
rep_seed = 1
model = FiniteResNetWithEmbedding(D, M, L, d_in = d_in, d_out=d_out, alpha=1.0, activation=linear, activation_der=linear_der, seed=rep_seed)
model.W_e = rootW_e[:D]
model.W_u = rootW_u[:D]
scale_eta_v, scale_eta_u = D, D
loss, Hs, Bs, model_outputs = train_track(model, x_true.reshape(-1,1), y_true.reshape(-1,1), K, eta_0, scale_eta_u, scale_eta_v)

In [ ]:
with open(f"data/exp_track_single_trajectory/model_outputs_D{D}_L{L}_M{M}.pkl", "wb") as f:
    pickle.dump(model_outputs, f)
with open(f"data/exp_track_single_trajectory/hs_D{D}_L{L}_M{M}.pkl", "wb") as f: 
    pickle.dump(Hs, f)
with open(f"data/exp_track_single_trajectory/bs_D{D}_L{L}_M{M}.pkl", "wb") as f: 
    pickle.dump(Bs, f)